In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [92]:
import math

from typing import List

class SiLU(nn.Module):
    def forward(self, x):
        return x * F.sigmoid(x)

class SwiGLU(nn.Module):
    def __init__(self, emb_size: int, dropout: float =0.1):
        super().__init__()
        self.emb_size = emb_size
        self.dropout = dropout

        self.gate = nn.Linear(self.emb_size, 4*self.emb_size)
        self.up = nn.Linear(self.emb_size, 4*self.emb_size)
        self.down = nn.Linear(4*self.emb_size, self.emb_size)
        self.silu = SiLU()
        self.drpt = nn.Dropout(dropout)

    def forward(self, x):
        return self.drpt(self.down(self.up(x) * self.silu(self.gate(x))))

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps=1e-6):
        super().__init__()
        self.dim = dim
        self.eps = eps

        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        a = torch.sqrt(torch.mean(x ** 2, 2,keepdim=True) + self.eps)
        b = x/a
        return torch.mul(self.weight , b)


class RoPE(nn.Module):
    def __init__(self, head_size: int, max_seq_len: int, base: int =10_000):
        super().__init__()
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.base = base

        theta = 1 / torch.Tensor([self.base]) ** (torch.Tensor([i for i in range(0, self.head_size, 2)]) / self.head_size)
        positional_indexes = torch.arange(self.max_seq_len, dtype=torch.float32)
        frequency_matrix = positional_indexes.reshape(positional_indexes.shape[0], 1) * theta
        self.cosinus_matrix = torch.cos(frequency_matrix)
        self.sinus_matrix = torch.sin(frequency_matrix)

    def forward(self, x, start_pos=0):
        batch_size, seq_len, head_size = x.shape

        local_cosinus_matrix = self.cosinus_matrix[start_pos:start_pos + seq_len,  :]
        local_sinus_matrix = self.sinus_matrix[start_pos:start_pos + seq_len,  :]

        even = x[:,:,::2]
        odd = x[:,:,1::2]

        return_tensor = torch.zeros_like(x)

        return_tensor[:,:, ::2] = even * local_cosinus_matrix - odd * local_sinus_matrix
        return_tensor[:,:, 1::2] = odd * local_cosinus_matrix + even * local_sinus_matrix
                
        return return_tensor

class TokenEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, emb_size: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, emb_size)

    def forward(self, x: torch.Tensor):
        return self.embeddings(x)

import math 

class HeadAttention(nn.Module):
    def __init__(self, emb_size: int, head_size:int, max_seq_len: int, rope: RoPE):
        super().__init__()
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.rope: RoPE = rope

        self.W_k = nn.Linear(self.emb_size, self.head_size)
        self.W_q = nn.Linear(self.emb_size, self.head_size)
        self.W_v = nn.Linear(self.emb_size, self.head_size)

        self.mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len))

    def forward(self, x: torch.Tensor,
                 use_cache: bool = True, cache: tuple = None):
        batch_size, seq_len, emb_size = x.shape
        key = self.W_k(x)
        query = self.W_q(x)
        value = self.W_v(x)

        if cache is not None:
            start_pos = cache[0].shape[1]
            key = self.rope(key, start_pos)
            query = self.rope(query, start_pos)
        else:
            key = self.rope(key)
            query = self.rope(query)

        
        if cache is not None:
            key = torch.cat([cache[0], key], 1)
        if cache is not None:
            value = torch.cat([cache[1], value], 1)
            
        
        attention_matrix : torch.Tensor = query @ key.transpose(1,2) / math.sqrt(self.head_size)

        if cache is None:
            mask = self.mask[:seq_len, :seq_len] == 0
            attention_matrix = attention_matrix.masked_fill(mask, float('-inf'))
        
        attention_matrix = torch.softmax(attention_matrix, 2) 

        x = attention_matrix @ value
        
        if not use_cache:  
            return (x, None)

        if use_cache:
            return (x, (key, value))


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, rope: RoPE, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.rope = rope

        self.head_attentions = nn.ModuleList([HeadAttention(self.emb_size, self.head_size, self.max_seq_len, rope) for i in range(self.num_heads)])
        self.otpt = nn.Linear(self.head_size * self.num_heads, self.emb_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, use_cache: bool=True, cache=None):
        if cache is not None:    
            tensors_and_caches = [head_attention(x, use_cache, ch) for ch, head_attention in zip(cache, self.head_attentions)]
        else:
            tensors_and_caches = [head_attention(x, use_cache, cache) for head_attention in self.head_attentions]
        tensors = [ tensor for tensor, _ in tensors_and_caches]
        cache_new = [cache for _, cache in tensors_and_caches]
            
        tensor = torch.cat(tensors, dim=2)
        tensor = self.otpt(tensor)
        tensor = self.dropout(tensor)

        if use_cache:
            return (tensor, cache_new)

        return (tensor, None)

class Decoder(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, rope: RoPE, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        self.multi_head_attention = MultiHeadAttention(self.num_heads, self.emb_size, self.head_size, self.max_seq_len, rope, self.dropout)
        self.feed_forward = SwiGLU(self.emb_size, self.dropout)
        self.first_norm = RMSNorm(self.emb_size)
        self.second_norm = RMSNorm(self.emb_size)

    def forward(self, x: torch.Tensor, use_cache = True, cache = None):
        tensor = self.first_norm(x)
        tensor, cache = self.multi_head_attention(tensor, use_cache, cache)
        tensor = x + tensor
        tensor = tensor + self.feed_forward(self.second_norm(tensor))
        if use_cache:
            return (tensor, cache)

        return (tensor, None)


class Llama(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, emb_size: int,
                num_heads: int, head_size: int, num_layers: int,
                dropout: float = 0.1, device: str = "cpu"):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.head_size = head_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.device = device

        self.token_embeddings = TokenEmbeddings(self.vocab_size, self.emb_size)
        self.rope = RoPE(self.head_size, self.max_seq_len)
        #self.positional_embeddings = PositionalEmbeddings(self.max_seq_len, self.emb_size)
        self.drpt = nn.Dropout(self.dropout)
        self.decoders = [Decoder(self.num_heads, self.emb_size, self.head_size, self.max_seq_len, self.rope, self.dropout) for i in range(self.num_layers)]
        self.linear = nn.Linear(self.emb_size, self.vocab_size)
        self.norm = RMSNorm(self.emb_size)

    def forward(self, x: torch.Tensor, use_cache=True, cache=None):
        tensor = self.token_embeddings(x)
        tensor = self.drpt(tensor)
        new_caches = []
        if cache is not None:
            for ch, decoder in zip(cache,self.decoders):
                tensor, cache_decoder = decoder(tensor, use_cache, ch)
                new_caches.append(cache_decoder)
        else:
            for decoder in self.decoders:
                tensor, cache_decoder = decoder(tensor, use_cache, cache)
                new_caches.append(cache_decoder)
        tensor = self.norm(tensor)
        tensor = self.linear(tensor)
        if use_cache:
            return (tensor, new_caches)
        
        return (tensor, None) 
        
        
    def generate(self, x: torch.Tensor, max_new_tokens: int, do_sample: bool, temperature: float = 1.0,
                top_k: int = None, top_p: float = None, use_cache=True):

        for i in range(max_new_tokens):
            if i == 0:
                x_hat = x[:, :]
            else:
                x_hat = x[:, -1:]
            if i == 0:
                logits, cache = self.forward(x_hat, use_cache=use_cache, cache=None)
            else:
                logits, cache = self.forward(x_hat, use_cache=use_cache, cache=cache)
            logits = logits[:, -1, :]
            logits /= temperature
            if not do_sample:
                probas = logits.softmax(1)
                new_tokens = torch.argmax(probas, dim=1)
            else:
                if top_k is not None:
                    new_logits = []
                    for j in range(logits.shape[0]):
                        logits_j = logits[j,:]
                        topk_values, topk_indicies = torch.topk(logits_j, top_k)
                        mask = torch.ones_like(logits_j) == 1
                        mask[topk_indicies] = 0
                        logits_j.masked_fill_(mask, float('-inf'))
                        new_logits.append(logits_j)
                    logits = torch.stack(new_logits,)
                if top_p is not None:
                    new_logits = []
                    for j in range(logits.shape[0]):
                        logits_j = logits[j,:]
                        probabilities = logits_j.softmax(0)
                        
                        max_probabilities = sorted(probabilities.detach().numpy(),reverse=True)

                        cum_proba = max_probabilities[0]
                        min_proba = max_probabilities[0]
                        m = 1
                        while m < len(max_probabilities):
                            cum_proba += max_probabilities[m]
                            if cum_proba < top_p:
                                min_proba = max_probabilities[m]
                            else:
                                break
                            m += 1
                        mask = probabilities  < min_proba

                        logits_j.masked_fill_(mask, float('-inf'))
                        new_logits.append(logits_j)
                    logits = torch.stack(new_logits,)
                        
                probas = logits.softmax(1)
                new_tokens = torch.multinomial(probas, 1)
            new_tokens = torch.reshape(new_tokens, (new_tokens.shape[0], 1))
            x = torch.cat([x, new_tokens], dim=1)
        return x

    def save(self, path):
        torch.save({
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_seq_len': self.max_seq_len,
            'emb_size': self.emb_size,
            'num_heads': self.num_heads,
            'head_size': self.head_size,
            'num_layers': self.num_layers
        }, path)
        
    @classmethod
    def load(cls, path, device):
        checkpoint = torch.load(path, map_location=device)
        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_seq_len=checkpoint['max_seq_len'],
            emb_size=checkpoint['emb_size'],
            num_heads=checkpoint['num_heads'],
            head_size=checkpoint['head_size'],
            num_layers=checkpoint['num_layers']
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model
    
    def fit(self, train_loader , valid_loader, 
           num_epoch: int, learning_rate: float,
           use_cache=False):

        self.to(self.device)

        criterion = nn.CrossEntropyLoss()

        adam = torch.optim.Adam(self.parameters(), lr=learning_rate)

        for epoch in range(num_epoch):
            self.train()

            for inputs, targets in train_loader:
                batch_size, seq_len = inputs.shape
                outputs, cache = self.forward(inputs, use_cache=use_cache)

                outputs = outputs.reshape(batch_size * seq_len, self.vocab_size)
                targets = targets.reshape(batch_size * seq_len)

                self.loss = criterion(outputs, targets)

                adam.zero_grad()
                self.loss.backward()

                adam.step()

            self.eval()
    
            with torch.no_grad():
                for inputs, targets in valid_loader:
                    
                    batch_size, seq_len = inputs.shape
                    outputs, cache = self.forward(inputs, use_cache=use_cache)
    
                    outputs = outputs.reshape(batch_size * seq_len, self.vocab_size)
                    targets = targets.reshape(batch_size * seq_len)
    
                    self.loss = criterion(outputs, targets)



In [94]:
Llama(10, 15,5, 1, 8,1).forward(torch.LongTensor([[5,4,3,2,1],[2,3,4,5,6]]))[0]





tensor([[[-0.4962, -0.5998,  0.7865, -0.8738, -0.5109, -0.9833,  0.4749,
          -0.6287,  0.9885,  0.0238],
         [ 0.8560, -0.6101,  0.3452, -0.2918, -0.8992, -0.2712, -0.4573,
          -0.7882, -1.0289,  0.8855],
         [-0.1589, -0.7453,  0.6694, -0.8454, -0.9188, -0.9722, -0.1929,
          -0.3758,  0.5393,  0.4020],
         [ 0.1823, -0.7217, -0.1027, -0.9141, -0.2391,  0.3095, -0.2545,
          -0.2225,  0.4560, -0.0795],
         [-0.1237, -0.9840,  0.4961, -0.9483, -0.9655, -0.9044, -0.2776,
          -0.2807,  0.5744,  0.4356]],

        [[ 0.2682, -0.5622, -0.2538, -0.6455,  0.1292,  0.0854, -0.3032,
           0.2082,  0.0543, -0.0367],
         [ 0.3522, -0.5192,  0.7386, -0.4945, -0.9338, -0.5356,  0.0098,
          -1.0866, -0.2662,  0.6105],
         [ 0.8656, -0.6940,  0.1723, -0.4201, -0.8547,  0.0024, -0.5565,
          -0.7020, -0.8745,  0.7673],
         [-0.4060, -0.7303,  0.7179, -0.8455, -0.5348, -0.9870,  0.5160,
          -0.6916,  0.8359,  0.1190],

In [84]:
class RoPE(nn.Module):
    def __init__(self, head_size: int, max_seq_len: int, base: int =10_000):
        super().__init__()
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.base = base

        theta = 1 / torch.Tensor([self.base]) ** (torch.Tensor([i for i in range(0, self.head_size, 2)]) / self.head_size)
        positional_indexes = torch.arange(self.max_seq_len, dtype=torch.float32)
        frequency_matrix = positional_indexes.reshape(positional_indexes.shape[0], 1) * theta
        self.cosinus_matrix = torch.cos(frequency_matrix)
        self.sinus_matrix = torch.sin(frequency_matrix)

    def forward(self, x, start_pos=0):
        batch_size, seq_len, head_size = x.shape

        local_cosinus_matrix = self.cosinus_matrix[start_pos:start_pos + seq_len,  :]
        local_sinus_matrix = self.sinus_matrix[start_pos:start_pos + seq_len,  :]

        even = x[:,:,::2]
        odd = x[:,:,1::2]

        return_tensor = torch.zeros_like(x)

        return_tensor[:,:, ::2] = even * local_cosinus_matrix - odd * local_sinus_matrix
        return_tensor[:,:, 1::2] = odd * local_cosinus_matrix + even * local_sinus_matrix
                
        return return_tensor


In [85]:
RoPE(4, 5)(torch.Tensor([[[1,2,3,4] , [1,2,3,4] , [1,2,3,4] ], [[1,2,3,4] , [1,2,3,4] , [1,2,3,4]]]))

tensor([[[ 1.0000,  2.0000,  3.0000,  4.0000],
         [-1.1426,  1.9221,  2.9599,  4.0298],
         [-2.2347,  0.0770,  2.9194,  4.0592]],

        [[ 1.0000,  2.0000,  3.0000,  4.0000],
         [-1.1426,  1.9221,  2.9599,  4.0298],
         [-2.2347,  0.0770,  2.9194,  4.0592]]])

In [77]:
1/torch.FloatTensor([2]) ** torch.Tensor([1,2])

tensor([0.5000, 0.2500])